In [1]:
import pandas as pd

# 1. Load the Weseq files
weseq_clin = pd.read_csv('weseq_286_Clinical.csv')
weseq_mut = pd.read_csv('weseq_286_Mutation.csv')

# The 20 genes from the TCGA dataset
genes_tcga = [
    'IDH1', 'TP53', 'ATRX', 'PTEN', 'EGFR', 'CIC', 'MUC16', 'PIK3CA', 
    'NF1', 'PIK3R1', 'FUBP1', 'RB1', 'NOTCH1', 'BCOR', 'CSMD3', 
    'SMARCA4', 'GRIN2A', 'IDH2', 'FAT4', 'PDGFRA'
]

/tmp/ipykernel_17192/233716536.py:5: DtypeWarning: Columns (1,2,3,4,5,6,7,8,9,10,11,13,14,15,16,17,18,20,22,23,26,29,30,31,33,34,36,37,38,40,41,42,43,44,45,47,48,52,54,55,56,57,58,59,60,62,63,64,65,66,67,69,70,71,72,74,75,76,77,78,80,81,83,85,86,87,89,90,93,94,96,100,102,104,105,106,109,111,112,113,114,115,117,120,123,124,125,126,128,131,132,134,138,142,144,148,150,151,152,154,155,156,157,158,160,161,162,165,166,167,168,169,170,171,172,173,175,178,179,180,181,184,185,189,190,191,192,194,195,196,197,198,199,200,201,204,206,208,210,211,212,213,215,216,217,218,219,222,223,224,225,230,231,234,235,237,238,239,240,242,246,247,249,250,251,253,254,255,256,257,259,260,261,262,264,266,268,274,275,278,280,281,283,284,285) have mixed types. Specify dtype option on import or set low_memory=False.
  weseq_mut = pd.read_csv('weseq_286_Mutation.csv')


In [2]:
# 2. Process Mutations (Non-Binary)
# Transpose so samples are rows and genes are columns
weseq_mut_trans = weseq_mut.set_index('Gene_Name').transpose()

# Keep only the genes that match TCGA
weseq_mut_aligned = weseq_mut_trans[genes_tcga].copy()

# Fill missing values with 'Wildtype' to keep the column categorical
weseq_mut_categorical = weseq_mut_aligned.fillna('Wildtype')
weseq_mut_categorical = weseq_mut_categorical.reset_index().rename(columns={'index': 'CGGA_ID'})

In [3]:
# 3. Process Clinical Features
# Grade: LGG -> 0, GBM -> 1
weseq_clin['Grade_encoded'] = weseq_clin['Grade'].apply(lambda x: 1 if 'IV' in str(x) else 0)
# Gender: Male -> 0, Female -> 1
weseq_clin['Gender_encoded'] = weseq_clin['Gender'].map({'Male': 0, 'Female': 1})
# Race: Set to 2 (Asian)
weseq_clin['Race'] = 2

In [4]:
# 4. Merge Clinical and Mutation data
weseq_processed = pd.merge(
    weseq_clin[['CGGA_ID', 'Grade_encoded', 'Gender_encoded', 'Age', 'Race']], 
    weseq_mut_categorical, 
    on='CGGA_ID'
)

# 5. Final Rename and Reorder
weseq_final = weseq_processed.rename(columns={
    'Grade_encoded': 'Grade',
    'Gender_encoded': 'Gender',
    'Age': 'Age_at_diagnosis'
})

final_cols = ['CGGA_ID', 'Grade', 'Gender', 'Age_at_diagnosis', 'Race'] + genes_tcga
weseq_final = weseq_final[final_cols]

In [5]:
# 6. Save
weseq_final.to_csv('weseq_categorical_with_id_and_raceV1.csv', index=False)
print("Categorical file saved.")

Categorical file saved.
